# OplR Round 3 Mutation Analysis

Analyze mutations in round 3 OplR plasmids compared to reference template.

In [9]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from Bio import SeqIO
from notebooks.jacob.round3.util import reindex_circular_genbank, find_mutations, Mutation
import time
import pandas as pd
from io import StringIO
from app.helpers.sequence_util import allele_set_to_seq_id

def print_summary(results):
    # Display mutations for each file
    # Summary statistics
    print("\n=== SUMMARY ===")
    print(f"Total files: {len(results)}")
    print(f"Successful: {sum(1 for v in results.values() if v is not None)}")
    print(f"Failed: {sum(1 for v in results.values() if v is None)}")

    # Count mutations by type
    total_mutations = sum(len(v) for v in results.values() if v is not None)
    print(f"\nTotal mutations found: {total_mutations}")

    mutation_types = {}
    for mutations in results.values():
        if mutations:
            for mut in mutations:
                mutation_types[mut.mutation_type] = mutation_types.get(mut.mutation_type, 0) + 1

    print("\nBy type:")
    for mut_type, count in sorted(mutation_types.items()):
        print(f"  {mut_type}: {count}")
        
    print("\n=== DETAILED RESULTS ===")
    for filename, mutations in sorted(results.items()):
        print(f"\n{filename}:")
        if mutations is None:
            print("  FAILED TO PROCESS")
        elif len(mutations) == 0:
            print("  No mutations found")
        else:
            for mut in mutations[:5]:
                print(f"  {mut}")
            if len(mutations) > 5:
                print(f"  ... and {len(mutations) - 5} more")
# Process all genbank files
# Paths
def get_results(reference_path, genbank_dir):
    # Load reference
    reference = SeqIO.read(reference_path, 'genbank')
    print(f"Loaded reference: {reference.id}, length={len(reference.seq)}")

    results = {}

    genbank_files = sorted(genbank_dir.glob('*.gbk'))
    print(f"Found {len(genbank_files)} genbank files to process\n")

    for gbk_file in genbank_files:
        print(f"Processing {gbk_file.name}...", end=' ')
        start_time = time.time()
        
        try:
            # Load query sequence
            query = SeqIO.read(gbk_file, 'genbank')
            
            # Reindex to match reference
            reindexed_query = reindex_circular_genbank(reference, query)
            
            # Find mutations
            mutations = find_mutations(reference, reindexed_query)
            
            # Store results
            results[gbk_file.name] = mutations
            
            elapsed = time.time() - start_time
            print(f"found {len(mutations)} mutations ({len(mutations)} grouped) ({elapsed:.2f}s)")
            
        except Exception as e:
            print(f"ERROR: {e}")
            import traceback
            traceback.print_exc()
            results[gbk_file.name] = None
    return results

def analyze_results(results, well_mapping_series):
    teselagen_mapping = {}
    for csv_file in Path('notebooks/jacob/round3/teselagen_id_mapping').glob('*.csv'):
        df = pd.read_csv(csv_file)
        for _, row in df.iterrows():
            teselagen_id = row['teselagen_seq_id']
            seq_id = row['seq_id']
            teselagen_mapping[teselagen_id] = {
                'intended_plasmid_id': teselagen_id,  # Use teselagen_id directly as plasmid_id
                'intended_seq_id': seq_id
            }

    print(f"Loaded {len(teselagen_mapping)} teselagen mappings")
    print(f"Sample keys: {list(teselagen_mapping.keys())[:3]}")

    well_mapping_series = well_mapping_series.apply(lambda x: f'{x[0]}{int(x[1:]):02d}')

    # Build analysis dataframe
    analysis_data = []

    for filename, mutations in results.items():
        if mutations is None:
            continue
        
        # Extract teselagen_id from filename
        # "LY6YL6_10_pAP_OplR_CH_R3_0010.gbk" -> "AP_OplR_CH_R3_0010"
        # Remove .gbk and find where "pAP" appears, then replace with "AP"
        base = filename.replace('.gbk', '')
        
        # Find the pAP_OplR part and convert to AP_OplR
        teselagen_id = '_'.join(base.split('_')[2:])[1:]
        
        if teselagen_id not in teselagen_mapping:
            print(f"Warning: {teselagen_id} not found in mapping (from {filename})")
            continue
        
        mapping = teselagen_mapping[teselagen_id]
        
        # Get CDS mutations
        cds_mutations = [m for m in mutations if m.amino_acid_change is not None]
        
        # Extract allele set from CDS mutations (excluding ? mutations)
        allele_set = set()
        for mut in cds_mutations:
            aa_change = mut.amino_acid_change
            # if aa_change and '?' not in aa_change and 'frameshift' not in aa_change.lower():
            allele_set.add(aa_change)
        
        # Convert to seq_id
        try:
            found_seq_id = allele_set_to_seq_id(allele_set)
        except Exception as e:
            found_seq_id = '_'.join(allele_set)
        
        # Check if it matches intended
        is_match = (found_seq_id == mapping['intended_seq_id'])
        
        # Get non-CDS mutations
        non_cds_mutations = [m for m in mutations if m.amino_acid_change is None]
        other_mutations_str = '; '.join([str(m) for m in non_cds_mutations]) if non_cds_mutations else ''

        intended_plasmid_id = '_'.join(mapping['intended_plasmid_id'].split('_')[:-1]) + '_' + mapping['intended_seq_id']
        
        analysis_data.append({
            'intended_plasmid_id': intended_plasmid_id,
            'teselagen_id': teselagen_id,
            'well': well_mapping_series['p' + teselagen_id],
            'intended_seq_id': mapping['intended_seq_id'],
            'found_seq_id': found_seq_id,
            'is_a_match': is_match,
            'other_mutations': other_mutations_str
        })

    # Create dataframe
    analysis_df = pd.DataFrame(analysis_data)
    analysis_df.sort_values(by='well', inplace=True)

    print(f"\nCreated analysis dataframe with {len(analysis_df)} rows")
    print(f"Matches: {analysis_df['is_a_match'].sum()} / {len(analysis_df)}")
    print(f"Match rate: {100 * analysis_df['is_a_match'].sum() / len(analysis_df):.1f}%")
    return analysis_df

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
oplr_reference_path = Path('notebooks/jacob/round1/template_plasmids/real_oplr_wt.gb')
oplr_genbank_dir = Path('notebooks/jacob/round3/251009_oplr_genbank-files')
oplr_results = get_results(oplr_reference_path, oplr_genbank_dir)   

print(f"\nProcessed {len(oplr_results)} files")


oplr_well_mapping = '''Construct	Well
pAP_OplR_CH_R3_0001	A1
pAP_OplR_CH_R3_0002	A2
pAP_OplR_CH_R3_0003	A3
pAP_OplR_CH_R3_0004	A4
pAP_OplR_CH_R3_0005	A5
pAP_OplR_CH_R3_0006	A6
pAP_OplR_CH_R3_0007	A7
pAP_OplR_CH_R3_0008	A8
pAP_OplR_CH_R3_0009	A9
pAP_OplR_CH_R3_0010	A10
pAP_OplR_CH_R3_0011	A11
pAP_OplR_CH_R3_0012	A12
pAP_OplR_CH_R3_0013	B1
pAP_OplR_CH_R3_0014	B2
pAP_OplR_CH_R3_0015	B3
pAP_OplR_CH_R3_0016	B4
pAP_OplR_CH_R3_0017	B5
pAP_OplR_CH_R3_0018	B6
pAP_OplR_CH_R3_0019	B7
pAP_OplR_CH_R3_0020	B8
pAP_OplR_CH_R3_0021	B9
pAP_OplR_CH_R3_0022	B10
pAP_OplR_CH_R3_0023	B11
pAP_OplR_CH_R3_0024	B12
pAP_OplR_DNL_R3_0001	C1
pAP_OplR_DNL_R3_0002	C2
pAP_OplR_DNL_R3_0003	C3
pAP_OplR_DNL_R3_0004	C4
pAP_OplR_DNL_R3_0005	C5
pAP_OplR_DNL_R3_0006	C6
pAP_OplR_DNL_R3_0007	C7
pAP_OplR_DNL_R3_0008	C8
pAP_OplR_DNL_R3_0009	C9
pAP_OplR_DNL_R3_0010	C10
pAP_OplR_DNL_R3_0011	C11
pAP_OplR_DNL_R3_0012	C12
pAP_OplR_DNL_R3_0013	D1
pAP_OplR_DNL_R3_0014	D2
pAP_OplR_DNL_R3_0015	D3
pAP_OplR_DNL_R3_0016	D4
pAP_OplR_DNL_R3_0017	D5
pAP_OplR_DNL_R3_0018	D6
pAP_OplR_DNL_R3_0019	D7
pAP_OplR_DNL_R3_0020	D8
pAP_OplR_DNL_R3_0021	D9
pAP_OplR_DNL_R3_0022	D10
pAP_OplR_DNL_R3_0023	D11
pAP_OplR_DNL_R3_0024	D12
pAP_OplR_EVL_R3_0001	E1
pAP_OplR_EVL_R3_0002	E2
pAP_OplR_EVL_R3_0003	E3
pAP_OplR_EVL_R3_0004	E4
pAP_OplR_EVL_R3_0005	E5
pAP_OplR_EVL_R3_0006	E6
pAP_OplR_EVL_R3_0007	E7
pAP_OplR_EVL_R3_0008	E8
pAP_OplR_EVL_R3_0009	E9
pAP_OplR_EVL_R3_0010	E10
pAP_OplR_EVL_R3_0011	E11
pAP_OplR_EVL_R3_0012	E12
pAP_OplR_EVL_R3_0013	F1
pAP_OplR_EVL_R3_0014	F2
pAP_OplR_EVL_R3_0015	F3
pAP_OplR_EVL_R3_0016	F4
pAP_OplR_EVL_R3_0017	F5
pAP_OplR_EVL_R3_0018	F6
pAP_OplR_EVL_R3_0019	F7
pAP_OplR_EVL_R3_0020	F8
pAP_OplR_EVL_R3_0021	F9
pAP_OplR_EVL_R3_0022	F10
pAP_OplR_EVL_R3_0023	F11
pAP_OplR_EVL_R3_0024	F12'''

oplr_well_mapping_df = pd.read_csv(StringIO(oplr_well_mapping), sep='\t')
print(oplr_well_mapping_df.columns)
oplr_well_mapping_series = oplr_well_mapping_df.set_index('Construct').Well

oplr_analysis_df = analyze_results(oplr_results, oplr_well_mapping_series)

# Show first few rows
# Save to CSV
output_path = Path('notebooks/jacob/round3/oplr_r3_sequencing_analysis.csv')
oplr_analysis_df.to_csv(output_path, index=False)
print(f"Saved analysis to {output_path}")

# Display full dataframe
oplr_analysis_df

Loaded reference: pAP_OplR_CH_R3_0001, length=6049
Found 66 genbank files to process

Processing LY6YL6_10_pAP_OplR_CH_R3_0010.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_13_pAP_OplR_CH_R3_0013.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_14_pAP_OplR_CH_R3_0014.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_15_pAP_OplR_CH_R3_0015.gbk... found 109 mutations (109 grouped) (0.02s)
Processing LY6YL6_16_pAP_OplR_CH_R3_0016.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_17_pAP_OplR_CH_R3_0017.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_18_pAP_OplR_CH_R3_0018.gbk... found 1146 mutations (1146 grouped) (0.06s)
Processing LY6YL6_19_pAP_OplR_CH_R3_0019.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_1_pAP_OplR_CH_R3_0001.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_20_pAP_OplR_CH_R3_0020.gbk... found 3 mutations (3 grouped) (0.02s)
Processing LY6YL6_21_pAP_OplR_CH_R3_0021.gbk.

,intended_plasmid_id,teselagen_id,well,intended_seq_id,found_seq_id,is_a_match,other_mutations
8,AP_OplR_CH_R3_V17A_E218L_Y255T,AP_OplR_CH_R3_0001,A01,V17A_E218L_Y255T,V17A_E218L_Y255T,True,
19,AP_OplR_CH_R3_K137A_S188L_Y255L,AP_OplR_CH_R3_0002,A02,K137A_S188L_Y255L,Q123D_S188L_Q303R,False,
29,AP_OplR_CH_R3_K137A_E218L_Y255T,AP_OplR_CH_R3_0003,A03,K137A_E218L_Y255T,K137A_E218L_Y255T,True,
40,AP_OplR_CH_R3_K137L_E218L_Y255T,AP_OplR_CH_R3_0004,A04,K137L_E218L_Y255T,W9L_K137L_E218L_Y255T,False,
48,AP_OplR_CH_R3_N139A_E218L_Y255T,AP_OplR_CH_R3_0005,A05,N139A_E218L_Y255T,N139A_E218L_Y255T,True,
...,...,...,...,...,...,...,...
57,AP_OplR_EVL_R3_Q123D_L241I_Q303R,AP_OplR_EVL_R3_0020,F08,Q123D_L241I_Q303R,Q123D_Q155S_Q303R,False,
58,AP_OplR_EVL_R3_Q123D_I252A_Q303R,AP_OplR_EVL_R3_0021,F09,Q123D_I252A_Q303R,Q123D_I252A_Q264K,False,
60,AP_OplR_EVL_R3_Q123D_Y255R_Q303R,AP_OplR_EVL_R3_0022,F10,Q123D_Y255R_Q303R,Q123D_T234N_A236A_G237S_A238S_C239E_G240H_L241...,False,
61,AP_OplR_EVL_R3_Q123D_L263R_Q296H,AP_OplR_EVL_R3_0023,F11,Q123D_L263R_Q296H,A3V_Q123D_L263R,False,


In [11]:
dbat_reference_path = Path('notebooks/jacob/round1/template_plasmids/GAH_DBAT.gb')
dbat_genbank_dir = Path('notebooks/jacob/round3/251009_dbat_genbank-files')
dbat_results = get_results(dbat_reference_path, dbat_genbank_dir)   

print(f"\nProcessed {len(dbat_results)} files")


dbat_well_mapping = '''Construct	Well
pGAH_DBAT_R3_0004	A1
pGAH_DBAT_R3_0005	A2
pGAH_DBAT_R3_0006	A3
pGAH_DBAT_R3_0007	A4
pGAH_DBAT_R3_0008	A5
pGAH_DBAT_R3_0009	A6
pGAH_DBAT_R3_0010	B1
pGAH_DBAT_R3_0011	B2
pGAH_DBAT_R3_0012	B3
pGAH_DBAT_R3_0013	B4
pGAH_DBAT_R3_0014	B5
pGAH_DBAT_R3_0015	B6
pGAH_DBAT_R3_0016	C1
pGAH_DBAT_R3_0017	C2
pGAH_DBAT_R3_0018	C3
pGAH_DBAT_R3_0019	C4
pGAH_DBAT_R3_0020	C5
pGAH_DBAT_R3_0021	C6
pGAH_DBAT_R3_0022	D1
pGAH_DBAT_R3_0023	D2
pGAH_DBAT_R3_0024	D3'''


dbat_well_mapping_df = pd.read_csv(StringIO(dbat_well_mapping), sep='\t')
dbat_well_mapping_series = dbat_well_mapping_df.set_index('Construct').Well

dbat_analysis_df = analyze_results(dbat_results, dbat_well_mapping_series)

# Show first few rows
# Save to CSV
dbat_output_path = Path('notebooks/jacob/round3/dbat_r3_sequencing_analysis.csv')
dbat_analysis_df.to_csv(dbat_output_path, index=False)
print(f"Saved analysis to {dbat_output_path}")

# Display full dataframe
dbat_analysis_df

/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqFeature.py:1043: BiopythonParserWarning: Attempting to fix invalid location '6052..398' as it looks like incorrect origin wrapping. Please fix input file, this could have unintended behavior.
  warnings.warn(


Loaded reference: urn.local...q-j87elhh, length=6445
Found 19 genbank files to process

Processing WM9MJ2_10_pGAH_DBAT_R3_0013.gbk... found 19 mutations (19 grouped) (0.09s)
Processing WM9MJ2_11_pGAH_DBAT_R3_0014.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_12_pGAH_DBAT_R3_0015.gbk... found 18 mutations (18 grouped) (0.08s)
Processing WM9MJ2_13_pGAH_DBAT_R3_0016.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_15_pGAH_DBAT_R3_0018.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_16_pGAH_DBAT_R3_0019.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_17_pGAH_DBAT_R3_0020.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_18_pGAH_DBAT_R3_0021.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_19_pGAH_DBAT_R3_0022.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_1_pGAH_DBAT_R3_0004.gbk... found 19 mutations (19 grouped) (0.08s)
Processing WM9MJ2_20_pGAH_DBAT_R3_0023.gbk... found 1

,intended_plasmid_id,teselagen_id,well,intended_seq_id,found_seq_id,is_a_match,other_mutations
9,GAH_DBAT_R3_S23T_S189I_Q208P_L420E,GAH_DBAT_R3_0004,A01,S23T_S189I_Q208P_L420E,S23T_S189I_Q208P_L420E,True,"Mutation(position=0, mutation_type='insertion'..."
12,GAH_DBAT_R3_S23T_S189I_Y209D_L420E,GAH_DBAT_R3_0005,A02,S23T_S189I_Y209D_L420E,S23T_S189I_Q208R_Y209I_Y210T_H211T_F212S_R213V...,False,"Mutation(position=0, mutation_type='insertion'..."
13,GAH_DBAT_R3_S23T_S189I_Y209P_L420E,GAH_DBAT_R3_0007,A04,S23T_S189I_Y209P_L420E,S23T_S189I_L420E,False,"Mutation(position=0, mutation_type='insertion'..."
14,GAH_DBAT_R3_S23T_S189I_Y210D_C238K,GAH_DBAT_R3_0008,A05,S23T_S189I_Y210D_C238K,S23T_S189I_C238K,False,"Mutation(position=0, mutation_type='insertion'..."
15,GAH_DBAT_R3_S23T_S189I_Y210D_L420E,GAH_DBAT_R3_0009,A06,S23T_S189I_Y210D_L420E,S23T_S189I_Y210D_L420E,True,"Mutation(position=0, mutation_type='insertion'..."
16,GAH_DBAT_R3_S23T_S189I_Y210E_L420E,GAH_DBAT_R3_0010,B01,S23T_S189I_Y210E_L420E,S23T_S189I_Y210E_L420E,True,"Mutation(position=0, mutation_type='insertion'..."
17,GAH_DBAT_R3_S23T_S189I_Y210N_L420E,GAH_DBAT_R3_0011,B02,S23T_S189I_Y210N_L420E,S23T_S189I_Y210N_L420E,True,"Mutation(position=0, mutation_type='insertion'..."
18,GAH_DBAT_R3_S23T_S189I_Y210P_L420E,GAH_DBAT_R3_0012,B03,S23T_S189I_Y210P_L420E,S23T_S189I_Y210P_L420E,True,"Mutation(position=0, mutation_type='insertion'..."
0,GAH_DBAT_R3_S23T_S189I_Y210S_L420E,GAH_DBAT_R3_0013,B04,S23T_S189I_Y210S_L420E,S23T_S189I_Y210P_L420E,False,"Mutation(position=0, mutation_type='insertion'..."
1,GAH_DBAT_R3_S23T_S189I_H211D_L420E,GAH_DBAT_R3_0014,B05,S23T_S189I_H211D_L420E,S23T_S189I_H211D_L420E,True,"Mutation(position=0, mutation_type='insertion'..."


other_mutations
Mutation(position=0, mutation_type='insertion', reference_base='', variant_base='G', annotations=['AmpR'], amino_acid_change=None, end_position=None); Mutation(position=0, mutation_type='insertion', reference_base='', variant_base='G', annotations=['AmpR'], amino_acid_change=None, end_position=None); Mutation(position=715, mutation_type='substitution', reference_base='C', variant_base='G', annotations=['ori'], amino_acid_change=None, end_position=None); Mutation(position=3114, mutation_type='substitution', reference_base='T', variant_base='T', annotations=[], amino_acid_change=None, end_position=None); Mutation(position=3904, mutation_type='substitution', reference_base='T', variant_base='C', annotations=['f1 ori'], amino_acid_change=None, end_position=None); Mutation(position=5017, mutation_type='substitution', reference_base='A', variant_base='A', annotations=['URA3 promoter'], amino_acid_change=None, end_position=None); Mutation(position=5064, mutation_type='deletion

In [19]:
for ii in range(len(dbat_analysis_df)):
    print(ii)
    for change in dbat_analysis_df['other_mutations'].iloc[ii].split(';'):
        print(change)

0
Mutation(position=0, mutation_type='insertion', reference_base='', variant_base='G', annotations=['AmpR'], amino_acid_change=None, end_position=None)
 Mutation(position=0, mutation_type='insertion', reference_base='', variant_base='G', annotations=['AmpR'], amino_acid_change=None, end_position=None)
 Mutation(position=715, mutation_type='substitution', reference_base='C', variant_base='G', annotations=['ori'], amino_acid_change=None, end_position=None)
 Mutation(position=3114, mutation_type='substitution', reference_base='T', variant_base='T', annotations=[], amino_acid_change=None, end_position=None)
 Mutation(position=3904, mutation_type='substitution', reference_base='T', variant_base='C', annotations=['f1 ori'], amino_acid_change=None, end_position=None)
 Mutation(position=5017, mutation_type='substitution', reference_base='A', variant_base='A', annotations=['URA3 promoter'], amino_acid_change=None, end_position=None)
 Mutation(position=5064, mutation_type='deletion', reference_b